# Tennis Court Keypoint Model — Training

Run each cell in order. Everything runs in the browser via Google Colab — no installs needed on your laptop.

**Before starting:** upload your labelled  (exported from ) and your images zip to Google Drive, then update the paths in Cell 2.

In [ ]:
# Cell 1 — Check GPU
import subprocess
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "No GPU — go to Runtime > Change runtime type > T4 GPU")
import torch
print(f"PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Cell 2 — Mount Drive and set paths
from google.colab import drive
drive.mount("/content/drive")

# ← UPDATE THESE to match where you put files in Drive
ANNOTATIONS_JSON = "/content/drive/MyDrive/tennis_training/annotations.json"
IMAGES_DIR       = "/content/drive/MyDrive/tennis_training/images"
OUT_DIR          = "/content/drive/MyDrive/tennis_training/checkpoints"

import os
os.makedirs(OUT_DIR, exist_ok=True)
print("Paths set.")

In [ ]:
# Cell 3 — Split annotations into train/val (80/20)
import json, random, os

with open(ANNOTATIONS_JSON) as f:
    data = json.load(f)

random.seed(42)
anns = data["annotations"].copy()
random.shuffle(anns)
id_to_img = {img["id"]: img for img in data["images"]}

cut = int(len(anns) * 0.8)
train_anns, val_anns = anns[:cut], anns[cut:]

def make_split(anns_subset, path):
    img_ids = {a["image_id"] for a in anns_subset}
    imgs = [id_to_img[i] for i in img_ids if i in id_to_img]
    with open(path, "w") as f:
        json.dump({"images": imgs, "annotations": anns_subset}, f)
    print(f"  {path}: {len(anns_subset)} samples")

make_split(train_anns, "/content/train.json")
make_split(val_anns,   "/content/val.json")
print(f"Total: {len(anns)} | Train: {len(train_anns)} | Val: {len(val_anns)}")

In [ ]:
# Cell 4 — Clone repo to get training code
!git clone --depth 1 https://github.com/sarperozkaynak/tennis_analysis /content/tennis_analysis 2>&1 | tail -3
import sys
sys.path.insert(0, "/content/tennis_analysis")

In [ ]:
# Cell 5 — Train!
# Adjust --epochs and --batch to your dataset size:
#   < 500 images  → --epochs 150 --batch 16
#   500-1000      → --epochs 100 --batch 32
#   > 1000        → --epochs  80 --batch 48
!python /content/tennis_analysis/training/train.py \
    --train-ann /content/train.json \
    --val-ann   /content/val.json \
    --img-dir   {IMAGES_DIR} \
    --out-dir   {OUT_DIR} \
    --epochs 100 \
    --batch 32

In [ ]:
# Cell 6 — Quick sanity check on a sample image
import sys
sys.path.insert(0, "/content/tennis_analysis")
import os, cv2, glob
from training.predict import CourtKeypointPredictor
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

model_path = os.path.join(OUT_DIR, "best_court_kp_model.pth")
predictor  = CourtKeypointPredictor(model_path)

# Pick a random validation image
with open("/content/val.json") as f:
    val_data = json.load(f)
sample_name = random.choice(val_data["images"])["file_name"]
frame = cv2.imread(os.path.join(IMAGES_DIR, sample_name))
kp    = predictor.predict(frame)

fig, ax = plt.subplots(1, 1, figsize=(12, 7))
ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
colors = plt.cm.tab20(np.linspace(0, 1, 14))
for i in range(0, len(kp), 2):
    x, y = kp[i], kp[i+1]
    ax.plot(x, y, "o", color=colors[i//2], markersize=8)
    ax.text(x+5, y-5, str(i//2), color="white", fontsize=8,
            bbox=dict(boxstyle="round,pad=0.1", facecolor=colors[i//2], alpha=0.7))
ax.set_title(f"Predicted keypoints: {sample_name}")
ax.axis("off")
plt.tight_layout()
plt.savefig("/content/keypoint_preview.png", dpi=120)
plt.show()
print("Preview saved to /content/keypoint_preview.png")

## After training

Download  from your Drive folder and put it in  in the repo.

Then update  as described in  — the 10-line snippet that swaps in the trained model.